# Experiment 1 — the benchmark

**Steps 4, 5 and 6 of 8 — portfolio, backtest, attribution — for one idea.**

**It produces** a book in `Portfolio/`, a track record in `Backtest/` and a breakdown of the
return in `Attribution/`. **It prevents** a good signal in a portfolio nobody could hold, paper
returns that real trading would have erased, and factor beta sold as alpha.

Experiment 1 is the **declared benchmark**: the yardstick every later experiment is measured
against. It is not a null. It is a real strategy with a real return, so beating it is a higher bar
than beating a no-model control — and **its rules freeze once `FINDINGS_1.md` reports**, because a
change to them invalidates every comparison in `RESULTS.md`. Improvements go into a new experiment.

The hypothesis is in [`BLUEPRINT_1.md`](BLUEPRINT_1.md), **written before this notebook's rule**.
The running log is [`JOURNAL_1.md`](JOURNAL_1.md); the results that survive are in
[`FINDINGS_1.md`](FINDINGS_1.md).

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells, or `git switch example` to read one that is already filled in.

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

**What this notebook does not do.** It does not download anything, profile the universe, or compute
a signal. A number about the data itself belongs in the Universe or Data stage — that separation is
what keeps every experiment comparable, because all of them read the identical panel.

**Four modules beside this notebook are shared by every experiment** — `securities_panel.py`,
`portfolio_construction.py`, `backtest_engine.py` and `attribution_analysis.py`, one per Lab
library — and no strategy column is named in any of them. The benchmark writes its loading steps
inline instead of importing the panel loader, because a baseline that cannot be read top to bottom
without chasing an import is a worse baseline.

## The section contract

Every experiment notebook has the same shape, so anyone who has read one can read all of them.
**Everything below section 2 is strategy-agnostic**, given the three objects that section produces.

| Section | Contains |
| --- | --- |
| 0 · Setup | paths, and **the strategy's columns** — the only strategy names in this notebook outside the rule |
| 1 · The panel | load the refined files and reshape them |
| 2 · The rule | selection, sizing, timing. **The one cell you write** |
| 2.1 · Invariants | what every rule must pass, whatever it is |
| 3 · Construction | the book, and the diagnostics a person would run it on |
| 4 · Backtest | one engine pass, guarded import, reports-and-skips without a licence |
| 5 · Attribution | where the return came from, guarded the same way |
| 6 · Verdict | what it concluded, **in words** |
| Handoff | what the next stage consumes, and what this one left open |

## 0 · Setup

Paths, and the strategy's columns. **Declare them here, not in a shared module**, so a signal never
becomes every later experiment's default without anyone deciding it. Read only the columns the
strategy consumes.

Three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Basis | Why |
| --- | --- | --- |
| Daily mark | dividend-and-split adjusted close | total-return valuation between rebalances |
| Fill | dividend-and-split adjusted VWAP | the price a trade actually gets |
| Commission | **unadjusted** VWAP | per-share cents ride on the unadjusted share count |

The provider returns the VWAP columns as null, which is why the Curator reconstructs them as `c_*`.
Never point anything at the provider's own VWAP columns.

## 1 · The panel

Load the refined files, resolve **one position per security**, and reshape to matrices.

A point-in-time universe contains renamed securities: two identifiers sharing one identity, each
carrying part of the history. Left alone they are two independent positions and the book
double-counts at the changeover. Key positions by a stable identity — an ISIN where the seed
carries one — falling back to the identifier itself, and where two legs overlap on a date let the
leg still reporting later win.

The long panel then becomes one wide `dates x securities` matrix per input, which is what makes the
whole rule in section 2 a handful of vectorised lines instead of a loop over files.

## 2 · The rule — the one cell you write

Three statements, in order: **who is eligible**, **how much of each**, and **when to trade**.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates x securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | a date index | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES x securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy. The residual becomes cash in section 3.1, parked in a real priced instrument,
because the engine's weight file has no cash row of its own.

**Sizing is a seam, not a decision buried in the rule.** Hand the eligible set and a returns
history that has already been cut off before today to a weighting function in
`portfolio_construction.py`, and swapping equal
weight for inverse volatility, a minimum-variance optimiser, hierarchical risk parity or a call
into the KaxaNuk Portfolio Construction library is one line. That is what makes two experiments
comparable rather than merely adjacent.

**Trade only when something changed.** A signal that has not moved is not a reason to pay
commission.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t-1*, and the fill
happens at *t*'s price — a full day between the signal and the fill.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists.

## 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged**,
whatever the strategy is:

- no book is more than fully invested, and none is negatively invested;
- no negative weights, if the strategy is long-only;
- **every security paid for today had its signal on at the prior close** — check the signal itself,
  not the composed eligibility, because the signal is the thing that had to exist in advance;
- every security bought is tradable on the day it is bought, so a fill price exists;
- nothing is still held on a day after it stopped being tradable.

## 3 · Construction — is this a book you would actually run?

**This is where step 4, Portfolio Construction, lives.** Four properties, each with a failure mode
a performance chart would hide:

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Measure turnover **target-to-target**. The realised figure is lower, because between rebalances the
winners drift up on their own; that calculation needs drifted weights and belongs to the backtest.

> **The blueprint's predictions about the *shape* of the book, rather than about its return, are
> settled here — before any backtest.** They are the first ones that can be wrong, and the cheapest
> to be wrong about.

## 3.1 · Write the deliverables

Two views of the same book, because two readers need it: a **long, human-readable** one with names
and classifications attached, and a **wide, identifier-keyed** `portfolio_weights.csv` — the
backtest engine's input, which looks each identifier up in the market-data folder and so has to
speak in identifiers, not in stitched positions.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; here the residual becomes a weight in the cash proxy, so the engine charges commission on
going to cash and earns the yield while there. A strategy whose defining move is *sell everything*
has to pay for it.

## 4 · Backtest — KaxaNuk Backtest Engine

**In plain words:** run the rules over history, with costs, without peeking ahead.

The weight file goes to the licensed engine, which simulates the book share by share: it fills at a
real price, charges per-share commission on the unadjusted price, holds integer share counts and a
cash reserve, marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository**, and results are accepted **net** or not at all.
Clip the window to the shortest benchmark up front rather than discovering it as a crash, and report
which benchmark bound it.

> **Guard the import.** The engine installs from KaxaNuk's licensed index rather than PyPI, so this
> section reports what is missing and skips without it. Everything in `Portfolio/` is already
> written and does not depend on the engine — a clone with no licence gets a real book and no
> numbers, by design.

## 5 · Attribution — KaxaNuk Attribution Analysis

**In plain words:** which part of the return did you actually earn?

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler**, the first cut: active return into an **allocation** effect — being
  overweight the right groups — and a **selection** effect, picking the right securities inside
  them. The exact lever that moved.
- **A factor model**, the second layer: excess return into **compensated factor tilts** — beta,
  momentum, residual volatility, liquidity — and **idiosyncratic** alpha, what was earned on
  purpose rather than by accident.
- **Brinson-Fachler again, on the residual**, the third pass: the selection story sharpens, and
  it says whether the Sharpe survives once the factor turns.

**What it settles and what it does not** is in [`../../AGENTS.md`](../../AGENTS.md) — including the
four counterfactual books that answer what the factor model cannot.

**Two inputs are supplied by hand**, from `Data/Curator/Benchmarks/` and `Data/Curator/Factors/` —
the benchmark's weights and returns, and the factor returns. No price provider sells them. Getting
their layout wrong makes the loader read the attribution transposed rather than fail, so shape them
in one place and say what is missing before trying.

**What binds the window.** The attribution period is the intersection of the factor files and the
benchmark holdings, so it is usually *shorter* than the backtest. The two sets of numbers describe
different periods and must not be compared directly. Record both windows in `FINDINGS_1.md`.

## 6 · Verdict

**In words.** A notebook that ends in a number and no sentence gets read as whatever the reader
hoped.

Three sentences: does the book work; what attribution says about why; what the next experiment
should change. Then copy the numbers into [`FINDINGS_1.md`](FINDINGS_1.md) — **that file is the
record, this notebook is the method.**

If sections 4 and 5 reported "not installed", this notebook has produced a book and no result,
which is the honest outcome and not a failure.

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine and the attribution library |
| `Portfolio/` — the readable book and the summaries | humans, and `FINDINGS_1.md` |
| `Backtest/`, `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against this benchmark meaningful.

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** | The book is real; the performance is not measured, and the blueprint's return predictions stay open. |
| 2 | **Turnover is target-to-target, not realised.** | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by the supplied files' coverage. | The two sets of numbers describe different periods. |
| 4 | **Delisting exits use one day of hindsight.** | Inert on a universe of live securities; load-bearing on any universe that retains delisted names. |
| 5 | **Cash is a real instrument**, so going flat costs commission and earns a yield. | A strategy that trades to cash often is partly a bet on the front end of the curve. Ask attribution about it. |
| 6 | **Every lever the benchmark declines** — a weight cap, a minimum holding count, risk-aware sizing — is a later experiment, and each has to beat this book to earn its place. | Complexity is added one lever at a time. |